In [302]:
# !pip install LunarCalendar

In [303]:
import streamlit as st
import pandas as pd
from datetime import datetime, timedelta, date
from itertools import cycle
from PIL import Image
import re
import matplotlib.pyplot as plt
import numpy as np
import csv
import os

animal = ["крыса", "бык", "тигр", "кролик", "дракон", "змея", "лошадь", "коза", "обезьяна", "петух", "собака", "свинья"]
stihiya = ["дерево", "дерево", "огонь", "огонь", "почва", "почва",  "металл",  "металл", "вода", "вода"]
in_yan = ["ян", "инь"]

vis_yaer = [1920, 1924, 1928, 1932, 1936, 1940, 1944, 1948, 1952, 1956, 1960, 1964, 1968, 
            1972, 1976, 1980, 1984, 1988, 1992, 1996, 2000, 2004, 2008, 2012, 2016, 2020, 
            2024, 2028, 2032, 2036, 2040, 2044, 2048, 2052]

moon_palace = dict({1: [1920, 1942, 0, 1987, 2009, 2032], 2: [0, 1943, 1965, 1988, 2010, 0], 
    3: [1921, 1944, 1966, 0, 2011, 2033], 4: [1922, 0, 1967, 1989, 2012, 2034], 
    5: [1923, 1945, 1968, 1990, 0, 2035], 6: [1924, 1946, 0, 1991, 2013, 2036], 
    7: [0, 1947, 1969, 1992, 2014, 0], 8: [1925, 1948, 1970, 0, 2015, 2037], 
    9: [1926, 0, 1971, 1993, 2016, 2038], 10: [1927, 1949, 1972, 1994, 0, 2039], 
    11: [1928, 1950, 0, 1995, 2017, 2040], 12: [0, 1951, 1973, 1996, 2018, 0], 
    13: [1929, 1952, 1974, 0, 2019, 2041], 14: [1930, 0, 1975, 1997, 2020, 2042], 
    15: [1931, 1953, 1976, 1998, 0, 2043], 16: [1932, 1954, 0, 1999, 2021, 2044], 
    17: [0, 1955, 1977, 2000, 2022, 0], 18: [1933, 1956, 1978, 0, 2023, 2045], 
    19: [1934, 0, 1979, 2001, 2024, 2046], 20: [1935, 1957, 1980, 2002, 0, 2047], 
    21: [1936, 1958, 0, 2003, 2025, 2048], 22: [0, 1959, 1981, 2004, 2026, 0], 
    23: [1937, 1960, 1982, 0, 2027, 2049], 24: [1938, 0, 1983, 2005, 2028, 2050], 
    25: [1939, 1961, 1984, 2006, 0, 2051], 26: [1940, 1962, 0, 2007, 2029, 2052], 
    27: [0, 1963, 1985, 2008, 2030, 0], 28: [1941, 1964, 1986, 0, 2031, 2053]})

sec_step = {1: 27,
            2: 2,
            3: 2,
            4: 5,
            5: 7,
            6: 10,
            7: 12,
            8: 15,
            9: 18,
            10: 20,
            11: 23,
            12: 25}


man = ["Liv.1", "Liv.4", "Liv.3", "Gb.37/Liv.3", "Liv.5/Gb.40", "Liv.2", "Liv.8", 
       "Kid.1", "Kid.7", "Kid.3", "Bl.58/Kid.3", "Kid.4/Bl.64", "Kid.2", "Kid.10", 
       "Lu.11", "Lu.8", "Lu.9", "Co.6/Lu.9", "Co.4/Lu.7", "Lu.10", "Lu.5", 
       "Ht.9/Hg.9", "Ht.4/Hg.5", "Ht.7/Hg.7", "Si.7/Ht.7/Hg.7", 
       "Ht.5/Hg.6/Si.4", "Ht.8/Hg.8", "Ht.3/ Hg.3"]

woman = ["Gb.41", "Gb.44", "Gb.34", "Gb.37/Liv.3", "Liv.5/Gb.40", "Gb.38", "Gb.43", 
       "Bl.65", "Bl.67", "Bl.40", "Bl.58/Kid.3", "Kid.4/Bl.64", "Bl.60", "Bl.66", 
       "Co.3", "Co.1", "Co.11", "Co.6/Lu.9", "Co4/Lu.7", "Co.5", "Co.2", 
       "Si.3", "Si.1", "Si.8", "Si.7/Ht.7/Hg.7", "Ht.5/Hg.6/Si.4", "Si.5", "Si.2"]

sky = {'甲': ':green[甲]',
        '乙': ':green[乙]',
        '丙': ':red[丙]',
        '丁': ':red[丁]',
        '戊': ':orange[戊]',
        '己': ':orange[己]',
        '庚': ':darkgray[庚]',
        '辛': ':darkgray[辛]',
        '壬': ':blue[壬]',
        '癸': ':blue[癸]'}

earth = {'子': ':blue[子]',
        '丑': ':orange[丑]',
        '寅': ':green[寅]',
        '卯': ':green[卯]',
        '辰': ':orange[辰]',
        '巳': ':red[巳]',
        '午': ':red[午]',
        '未': ':orange[未]',
        '申': ':darkgray[申]',
        '酉': ':darkgray[酉]',
        '戌': ':orange[戌]',
        '亥': ':blue[亥]'}

def read_files():       
    cities = pd.read_csv("data/cities.csv")
    return cities


# def get_year(our_date):
#     stih = cycle(stihiya)
#     anim = cycle(animal)
#     inyan = cycle(in_yan)
#     start_year = 1923
#     year_china = our_date.year
#     if our_date.month < 2: 
#         year_china = year_china-1
#     lst = []
#     for i in range(year_china - start_year):
#         start_year+=1
#         lst.append(f"{next(anim)} {next(inyan)} {next(stih)}".capitalize())
#     return lst[-1]


# def get_month(our_date):
#     stih = cycle(stihiya)
#     anim = cycle(animal)
#     inyan = cycle(in_yan)  
#     start_month = date(1924, 1, 1)
#     end_months=(our_date.year-start_month.year)*12 + our_date.month
#     lst = []
#     for i in range(end_months):
#         lst.append(f"{next(anim)} {next(inyan)} {next(stih)}".capitalize())
#     return lst[-1]


# def get_day(our_date):
#     stih = cycle(stihiya)
#     anim = cycle(animal)
#     inyan = cycle(in_yan)  
#     start_day = date(1923, 2, 20)
#     lst = []
#     for i in range((our_date - start_day).days+1):
#         lst.append(f"{next(anim)} {next(inyan)} {next(stih)}".capitalize())
#     return lst[-1]


In [304]:
# Обработка интересующей даты

our_date =  '06/04,2025' #input('Введите дату') # 

our_date = vis_date = re.sub('\D', '.', our_date)
our_date = our_date.split('.')
d = int(our_date[0])
m = int(our_date[1])
y = int(our_date[2])

our_date = date(y, m, d)
print("our_date:", our_date)
print(vis_date)



our_date: 2025-04-06
06.04.2025


In [305]:
# # Вычисляем дату наступления нового года по китайскому календарю
# import datetime
# from lunarcalendar import Converter, Solar, Lunar, DateNotExist

# l = Lunar(year=our_date.year, month=1, day=1, isleap=False)
# china_new_year = Converter.Lunar2Solar(l).to_date()
# china_new_year

In [306]:
## Создание таблицы с датами
# calender = []
# for d in range(65000):
#     calender.append(date(1926, 1, 1) + timedelta(days=d))

# years = []
# months = []
# days = []
# for our_date in calender:
#     year = get_year(our_date)
#     month = get_month(our_date)
#     day = get_day(our_date)
#     years.append(year)
#     months.append(month)
#     days.append(day)

# df = pd.DataFrame({
#     "date":calender,
#     "years":years,
#     "months":months,
#     "days":days
# })
# df[-2:]

In [307]:
earth_legs = pd.read_csv("data/earth_legs.csv")
sky_hands = pd.read_csv("data/sky_hands.csv")
planets = pd.read_csv("data/planets.csv")
moon_palace_df = pd.read_csv("data/moon_palace.csv")

calender = pd.read_csv("data/calender.csv")
cicle = pd.read_csv("data/cicle.csv")
calender['date'] = pd.to_datetime(calender['date'])

In [308]:
year_v = calender[calender['date']==pd.to_datetime(our_date)]['years'].values[0]
month_v = calender[calender['date']==pd.to_datetime(our_date)]['months'].values[0]
day_v = calender[calender['date']==pd.to_datetime(our_date)]['days'].values[0]
day = cicle[cicle["Название_calender"] == day_v]["Название_Русский"].values[0]
day_iero = cicle[cicle["Название_calender"] == day_v]["Иероглиф"].values[0]
month_iero = cicle[cicle["Название_calender"] == month_v]["Иероглиф"].values[0]
year_iero = cicle[cicle["Название_calender"] == year_v]["Иероглиф"].values[0]


birthday_df = pd.DataFrame(columns=["День", "Месяц", "Год"])
birthday_df["День"] = [
    day,
    sky[day_iero[0]] + earth[day_iero[1]]
]
birthday_df["Месяц"] = [
    cicle[cicle["Название_calender"] == month_v]["Название_Русский"].values[0],
    sky[month_iero[0]] + earth[month_iero[1]]
]
birthday_df["Год"] = [
    cicle[cicle["Название_calender"] == year_v]["Название_Русский"].values[0],
    sky[year_iero[0]] + earth[year_iero[1]]
]

In [309]:
cities[:1]

,Индекс,Тип региона,Регион,Тип района,Район,Тип города,Город,Тип н/п,Н/п,Код КЛАДР,...,Уровень по ФИАС,Признак центра района или региона,Код ОКАТО,Код ОКТМО,Код ИФНС,Часовой пояс,Широта,Долгота,Федеральный округ,Население
0,385200.0,Респ,Адыгея,NaN,NaN,г,Адыгейск,NaN,NaN,100000200000,...,4: город,0,79403000000,7.970300e+10,107,UTC+3,44.878372,39.190172,Южный,12689


In [310]:
city = "Москва"
cities[cities["Регион"].str.contains(city, regex=True)]

,Индекс,Тип региона,Регион,Тип района,Район,Тип города,Город,Тип н/п,Н/п,Код КЛАДР,...,Уровень по ФИАС,Признак центра района или региона,Код ОКАТО,Код ОКТМО,Код ИФНС,Часовой пояс,Широта,Долгота,Федеральный округ,Население
506,101000.0,г,Москва,NaN,NaN,NaN,NaN,NaN,NaN,7700000000000,...,1: регион,0,45000000000,NaN,7700,UTC+3,55.753879,37.620373,Центральный,11514330


In [311]:
cities = pd.read_csv("data/cities.csv")

In [312]:

city = "Хабар" # input("Введите название Вашего населённого пункта").capitalize()
if len(cities[cities["Город"].str.contains(city, regex=True).fillna(False)]) != 0:
        raw = cities[cities["Город"].str.contains(city, regex=True).fillna(False)][["Индекс", "Тип региона", "Регион", "Тип района", 
                                                                            "Район", "Тип города", "Город", "Тип н/п", "Н/п"]].dropna(axis=1)
elif len(cities[cities["Регион"].str.contains(city, regex=True).fillna(False)]) != 0:
        raw = (cities[cities["Регион"].str.contains(city, regex=True).fillna(False)][["Индекс", "Тип региона", "Регион", 
            "Тип района",	"Район", "Тип города", "Город", "Тип н/п",	"Н/п"]].dropna(axis=1))
else:
    raw = (cities[cities["Н/п"].str.contains(city, regex=True).fillna(False)][["Индекс", "Тип региона", "Регион", 
            "Тип района",	"Район", "Тип города", "Город", "Тип н/п",	"Н/п"]].dropna(axis=1))

id_city = raw.index
long = cities.loc[id_city, "Долгота"].values[0]
h = int(long//15)
minutes = int(round((long/15 - h)*60))
minutes

timedelta(hours=h, minutes=minutes)

CURRENT_TIME = datetime.now().time().strftime("%H:%M")
CURRENT_TIME_SOLAR = (datetime.utcnow() + timedelta(hours=h, minutes=minutes)).time().strftime('%H:%M')

print(f"Текущее административное время: {CURRENT_TIME}")
print(f"Среднее солнечное время: {CURRENT_TIME_SOLAR}")

Текущее административное время: 09:43
Среднее солнечное время: 15:43


C:\Users\anast\AppData\Local\Temp\ipykernel_4852\1844120502.py:2: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  if len(cities[cities["Город"].str.contains(city, regex=True).fillna(False)]) != 0:
C:\Users\anast\AppData\Local\Temp\ipykernel_4852\1844120502.py:3: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  raw = cities[cities["Город"].str.contains(city, regex=True).fillna(False)][["Индекс", "Тип региона", "Регион", "Тип района",


In [330]:
datetime.now().time().strftime("%H:%M")

'09:45'

#### Первая строка

In [313]:
a = f"{vis_date} день: {day}"
print(a)

06.04.2025 день: Деревянная Змея


#### Вторая строка

In [314]:
in_yan = cicle[cicle['Название_calender'] == day_v]['инь_ян'].values[0]
b = f"День: {in_yan.capitalize()}"
print(b)

День: Инь


#### Третья строка

In [315]:
c = f"ЦзяЦзы дня: № {cicle[cicle['Название_calender'] == day_v]['Цзя_Цзы'].values[0]}"
print(c)

ЦзяЦзы дня: № 42


#### Четвёртая строка

In [316]:
moon_palace_df = pd.read_csv("data/moon_palace.csv")

# Считаем лунную стоянку
for k, v in moon_palace.items():
    if our_date.year in v:
        first_step = k

if (our_date.year in vis_yaer) & (pd.to_datetime(our_date) > pd.to_datetime(f"{our_date.year}-02-28")):
    first_step = first_step+1

lunar_day = first_step + sec_step[our_date.month]+ our_date.day

while lunar_day > 28:
    lunar_day+=-28

if lunar_day in range(1, 15):
    lunar_day_ton = lunar_day+14
else:
    lunar_day_ton = lunar_day-14


symbol = moon_palace_df[moon_palace_df["Лунный_день"]==lunar_day]["Иероглиф"].values[0]
val = moon_palace_df[moon_palace_df["Лунный_день"]==lunar_day]["Созвездие"].values[0]

point = moon_palace_df[moon_palace_df["Лунный_день"]==lunar_day][["Точка_Ду_май", "Название"]].values[0][0] + \
    " " + moon_palace_df[moon_palace_df["Лунный_день"]==lunar_day][["Точка_Ду_май", "Название"]].values[0][1]

print(f"Лунная стоянка: {str(lunar_day)} {symbol} {val.capitalize()}")
print("=="*50)
print(f"Точки 28 Лунных Стоянок (Ду май): {point}")
print("--"*50)
print("Техника 28 Лунных Стоянок")
print("--"*50)

print(f"Помочь выйти событию (седирование) \
        \nЯн: \t{man[lunar_day-1]}\
        \nИнь: \t{woman[lunar_day-1]}")

print(f"Заставийть выйти событие (тонизация) \
        \nЯн: \t{man[lunar_day_ton-1]}\
        \nИнь: \t{woman[lunar_day_ton-1]}")

Лунная стоянка: 4 房 Покои
Точки 28 Лунных Стоянок (Ду май): Du.16 Фэн-Фу
----------------------------------------------------------------------------------------------------
Техника 28 Лунных Стоянок
----------------------------------------------------------------------------------------------------
Помочь выйти событию (седирование)         
Ян: 	Gb.37/Liv.3        
Инь: 	Gb.37/Liv.3
Заставийть выйти событие (тонизация)         
Ян: 	Co.6/Lu.9        
Инь: 	Co.6/Lu.9


#### Пятая строка

In [317]:
seasons = pd.read_csv("data/seasons.csv")

In [318]:
# Определяем сезон по дате

if (pd.to_datetime(our_date) < pd.to_datetime(seasons[str(our_date.year)][0])) or (pd.to_datetime(our_date) > pd.to_datetime(seasons[str(our_date.year)][23])):
    season = seasons.iloc[23][["Символ", "Название", "Иероглиф",	"Месяц", "Точки_Жэнь_май",	"Название_точки"]].values
else:
    for d in range(24):
        if (pd.to_datetime(our_date) > pd.to_datetime(seasons[str(our_date.year)][d])) & (pd.to_datetime(our_date)<pd.to_datetime(seasons[str(our_date.year)][d+1])):
            season = seasons.iloc[d][["Символ", "Название", "Иероглиф",	"Месяц", "Точки_Жэнь_май",	"Название_точки"]].values
            break
print(f"Точки 24 Сезонов (Жэнь май)")
print(" ".join(season[-2:]).strip())

Точки 24 Сезонов (Жэнь май)
VC20  Хуа-гай


#### Шестая строка

In [319]:
dow_dict = {0:"Понедельник", 1:"Вторник", 
            2:"Среда", 3:"Четверг",
            4:"Пятница", 5:"Суббота", 6:"Воскресенье"}

print(f"День недели: {dow_dict[pd.to_datetime(our_date).day_of_week]}")

День недели: Воскресенье


#### Седьмая строка

In [320]:
planet = planets[planets['День_недели']==pd.to_datetime(our_date).day_of_week]['Планета'].values[0]

print(f"Планета-покровитель: {planet.capitalize()}")

Планета-покровитель: Солнце


#### Фей тен ба фа

In [321]:
feitenbafa = pd.read_csv("data/feitenbafa.csv")
for_feitenbafa = pd.read_csv("data/for_feitenbafa.csv")

day_predictions = feitenbafa.merge(for_feitenbafa.rename(columns={"Иероглиф":day_iero[0]}))
feitenbafa_day = day_predictions[[day_iero[0], 'Иероглиф',	'Время',	'Канал',	'Точки']]

print("ФЭЙ ТЭН БА ФА")


time_now = datetime.now()
current_time = time_now.strftime("%H:%M:%S")
print("Текущее время:", current_time)  
current_hour = re.search(r"(\d*)", current_time)[0]
print("Текущий час:", current_hour)  

# Определяем час по текущему времени.

if ((int(current_hour) >= 0) & (int(current_hour) < 1)) or (int(current_hour) >= 23):
    current_hour_china_list = feitenbafa_day.iloc[0].values
else:
    for h in range(1, 12):
        if (int(current_hour) >= (feitenbafa['Время_int'][h])) & (int(current_hour) < (feitenbafa['Время_int'][h+1])):
            current_hour_china_list = feitenbafa_day.iloc[h].values
            break
        
print("\n".join(current_hour_china_list))

feitenbafa_day_disp = feitenbafa_day.iloc[:, 1:].T
feitenbafa_day_disp.to_csv("feitenbafa_day_disp.csv", index=False)
feitenbafa_day_disp = pd.read_csv("feitenbafa_day_disp.csv", header=1)

ФЭЙ ТЭН БА ФА
Текущее время: 09:43:38
Текущий час: 09
辛
巳
9:00 - 11:00
Ду-май
Si.3 + Bl.62


#### ЛИН ГУЙ БА ФА

In [322]:
for_lin_gui_ba_fa = pd.read_csv("data/for_lin_gui_ba_fa.csv")
current_hour_china = ''.join(current_hour_china_list[:2])


linguibafa = []
for i in feitenbafa_day.index:
    summ = sky_hands[sky_hands['Иероглиф']==day_iero[0]]['i_day'].values[0] + \
                sky_hands[sky_hands['Иероглиф']==feitenbafa_day.iloc[i, 0]]['i_hour'].values[0] + \
                earth_legs[earth_legs['Иероглиф']==day_iero[1]]['j_day'].values[0] + \
                earth_legs[earth_legs['Иероглиф']==feitenbafa_day.iloc[i, 1]]['j_hour'].values[0]

    if cicle[cicle['Иероглиф']==day_iero]['инь_ян'].values[0] == 'ян':
        res = summ%9
        if res == 0:
            res = 9
    else:
        res = summ%6
        if res == 0:
            res = 6  
        
    linguibafa_lst = list(feitenbafa_day.iloc[i,:3].values)
    linguibafa_lst.extend(for_lin_gui_ba_fa[for_lin_gui_ba_fa['res']==res].values[0][1:])
    linguibafa.append(linguibafa_lst)

    
linguibafa_df = pd.DataFrame(
    data=linguibafa,
    columns=[feitenbafa_day.columns[0], feitenbafa_day.columns[1], feitenbafa_day.columns[2],"Канал", "Точка", "Название_точки"]
)


linguibafa_df[linguibafa_df.columns[1:]].T.to_csv("data/linguibafa_df_disp.csv", index=False)
linguibafa_df_disp = pd.read_csv("data/linguibafa_df_disp.csv", header=1)


linguibafa_current_hour = linguibafa_df[linguibafa_df['Иероглиф']==current_hour_china[1]]
" || ".join(linguibafa_current_hour.iloc[0,1:].values.tolist())

'巳 || 9:00 - 11:00 || Ян-вэй || Th.5 || Вай-Гуань'

In [323]:
path_tai = os.listdir("data/tai_yan_ba_fa/")

In [324]:
path_tai

['丁壬.csv', '丙辛.csv', '乙庚.csv', '戊癸.csv', '甲己.csv']

In [325]:
current_hour_china

'辛巳'

In [326]:
list_tai = os.listdir("data/tai_yan_ba_fa/")
for l in list_tai:
    if day_iero[0] in l:
        file=re.findall(f'(\w*{day_iero[0]}\w*.csv)', l)

tai_yan_ba_fa = pd.read_csv(f"data/tai_yan_ba_fa/{file[0]}")

current_hour_taiyan = tai_yan_ba_fa[tai_yan_ba_fa["0"]==current_hour_china[1]]

In [327]:
tai_yan_ba_fa[tai_yan_ba_fa["0"]==current_hour_china[1]].index
tacurrent_hour_taiyan = i_yan_ba_fa.iloc[tai_yan_ba_fa[tai_yan_ba_fa["0"]==current_hour_china[1]].index[0]:tai_yan_ba_fa[tai_yan_ba_fa["0"]==current_hour_china[1]].index[0] + 2]

NameError: name 'i_yan_ba_fa' is not defined

In [ ]:
da_syao = pd.read_csv("data/da_syao.csv")

current_hour_china_list = da_syao[current_hour_china[1]].to_list()
            
            
# " || ".join(current_hour_china_list[1:])

In [ ]:
current_hour_china[1]

'乙酉'

In [ ]:
current_hour_china_list

['17.00 - 19.00', 'солн время', 'VC17', 'Kid.3', 'Bl.58']